In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from scipy.signal import savgol_filter
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# テストパターン選択（1つだけTrueにして実行）
# ============================================================
PATTERN_A_ADD_D2     = True    # d2をLGBに追加
PATTERN_B_KNN_DIST   = False   # KNN距離を追加（KNN加重ではない）
PATTERN_C_BOTH       = False   # A + B 両方

pattern_name = ""
if PATTERN_C_BOTH:
    pattern_name = "d2追加 + KNN距離"
elif PATTERN_A_ADD_D2:
    pattern_name = "d2追加のみ"
elif PATTERN_B_KNN_DIST:
    pattern_name = "KNN距離のみ"
else:
    pattern_name = "元コード完全再現"

print("=" * 60)
print(f"🧪 テスト: {pattern_name}")
print("=" * 60)

# ============================================================
# 1. データ読み込み
# ============================================================
with open('data/train.csv', 'r', encoding='cp932', errors='replace') as f:
    train = pd.read_csv(f)
with open('data/test.csv', 'r', encoding='cp932', errors='replace') as f:
    test = pd.read_csv(f)
submit = pd.read_csv('data/sample_submit.csv', header=None)

train = train[train['樹種'] != 'ベイスギ'].reset_index(drop=True)

spec_cols = [c for c in train.columns
             if c not in ['sample number', 'species number', '樹種', '含水率']]
y_train_log = np.log1p(train['含水率'])
groups = train['species number']

wavenumbers = np.array([float(c) for c in spec_cols])
wavelengths = np.where(wavenumbers > 0, 10000000 / wavenumbers, 0)
idx_1940 = np.argmin(np.abs(wavelengths - 1940))
idx_1300 = np.argmin(np.abs(wavelengths - 1300))

def apply_snv(X):
    m = np.mean(X, axis=1, keepdims=True)
    s = np.std(X, axis=1, keepdims=True) + 1e-8
    return (X - m) / s

# ============================================================
# 2. CVループ
# ============================================================
gkf = GroupKFold(n_splits=5)
X_test_raw = test[spec_cols].values

final_lgb = np.zeros(len(test))
oof_lgb = np.zeros(len(train))
fold_rmses = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(
    train[spec_cols].values, y_train_log, groups
)):
    va_species = train.iloc[va_idx]['樹種'].unique()
    print(f"\n{'─'*55}")
    print(f"📁 Fold {fold+1}/5  (train:{len(tr_idx)}, valid:{len(va_idx)})")
    print(f"   検証樹種: {list(va_species)}")

    X_tr_raw = train[spec_cols].values[tr_idx]
    y_tr = y_train_log.iloc[tr_idx].values
    X_va_raw = train[spec_cols].values[va_idx]
    y_va = y_train_log.iloc[va_idx].values
    X_te_raw = X_test_raw.copy()

    # ── 前処理（元コードと同一）──
    snv_tr = apply_snv(X_tr_raw)
    snv_va = apply_snv(X_va_raw)
    snv_te = apply_snv(X_te_raw)

    d1_tr = savgol_filter(snv_tr, window_length=15, polyorder=2, deriv=1, axis=1)
    d1_va = savgol_filter(snv_va, window_length=15, polyorder=2, deriv=1, axis=1)
    d1_te = savgol_filter(snv_te, window_length=15, polyorder=2, deriv=1, axis=1)

    d2_tr = savgol_filter(snv_tr, window_length=11, polyorder=2, deriv=2, axis=1)
    d2_va = savgol_filter(snv_va, window_length=11, polyorder=2, deriv=2, axis=1)
    d2_te = savgol_filter(snv_te, window_length=11, polyorder=2, deriv=2, axis=1)

    # ── 元コード特徴量 ──
    ratio_tr = (X_tr_raw[:, idx_1940] / (X_tr_raw[:, idx_1300] + 1e-8)).reshape(-1, 1)
    ratio_va = (X_va_raw[:, idx_1940] / (X_va_raw[:, idx_1300] + 1e-8)).reshape(-1, 1)
    ratio_te = (X_te_raw[:, idx_1940] / (X_te_raw[:, idx_1300] + 1e-8)).reshape(-1, 1)

    std_tr = np.std(X_tr_raw, axis=1, keepdims=True)
    std_va = np.std(X_va_raw, axis=1, keepdims=True)
    std_te = np.std(X_te_raw, axis=1, keepdims=True)

    # ── PCA + KNN（元コードと同一）──
    pca = PCA(n_components=10, random_state=42)
    pca_tr = pca.fit_transform(snv_tr)
    pca_va = pca.transform(snv_va)
    pca_te = pca.transform(snv_te)

    knn = NearestNeighbors(n_neighbors=5, metric='cosine')
    knn.fit(pca_tr)

    dist_tr, ind_tr = knn.kneighbors(pca_tr, n_neighbors=6)
    knn_ymean_tr = np.mean(y_tr[ind_tr[:, 1:]], axis=1).reshape(-1, 1)

    dist_va, ind_va = knn.kneighbors(pca_va, n_neighbors=5)
    knn_ymean_va = np.mean(y_tr[ind_va], axis=1).reshape(-1, 1)

    dist_te, ind_te = knn.kneighbors(pca_te, n_neighbors=5)
    knn_ymean_te = np.mean(y_tr[ind_te], axis=1).reshape(-1, 1)

    # ── LGB入力の組み立て ──
    # ベース（元コードと同一）
    parts_tr = [snv_tr, d1_tr, pca_tr, knn_ymean_tr, ratio_tr, std_tr]
    parts_va = [snv_va, d1_va, pca_va, knn_ymean_va, ratio_va, std_va]
    parts_te = [snv_te, d1_te, pca_te, knn_ymean_te, ratio_te, std_te]

    # パターンA or C: d2追加
    if PATTERN_A_ADD_D2 or PATTERN_C_BOTH:
        parts_tr.append(d2_tr)
        parts_va.append(d2_va)
        parts_te.append(d2_te)

    # パターンB or C: KNN距離
    if PATTERN_B_KNN_DIST or PATTERN_C_BOTH:
        knn_dist_tr = np.mean(dist_tr[:, 1:], axis=1).reshape(-1, 1)
        knn_dist_va = np.mean(dist_va, axis=1).reshape(-1, 1)
        knn_dist_te = np.mean(dist_te, axis=1).reshape(-1, 1)
        parts_tr.append(knn_dist_tr)
        parts_va.append(knn_dist_va)
        parts_te.append(knn_dist_te)

    feat_tr = np.hstack(parts_tr)
    feat_va = np.hstack(parts_va)
    feat_te = np.hstack(parts_te)

    if fold == 0:
        print(f"\n  📐 LGB入力次元: {feat_tr.shape[1]}")
        base_dim = snv_tr.shape[1] + d1_tr.shape[1] + 10 + 1 + 1 + 1
        print(f"     ベース(元コード): {base_dim}")
        if PATTERN_A_ADD_D2 or PATTERN_C_BOTH:
            print(f"     + d2:            {d2_tr.shape[1]}")
        if PATTERN_B_KNN_DIST or PATTERN_C_BOTH:
            print(f"     + KNN距離:       1")

    # ── LightGBM（元コードと同一パラメータ）──
    lgb_model = lgb.LGBMRegressor(
        n_estimators=1000, learning_rate=0.03, max_depth=5, num_leaves=31,
        subsample=0.8, colsample_bytree=0.3, random_state=42, verbosity=-1
    )
    lgb_model.fit(
        feat_tr, y_tr,
        eval_set=[(feat_va, y_va)],
        callbacks=[lgb.early_stopping(30, verbose=False)]
    )
    p_va = np.expm1(lgb_model.predict(feat_va))
    p_te = np.expm1(lgb_model.predict(feat_te))

    oof_lgb[va_idx] = p_va
    final_lgb += p_te / 5

    y_va_real = np.expm1(y_va)
    rmse = np.sqrt(mean_squared_error(y_va_real, p_va))
    fold_rmses.append(rmse)
    print(f"  🌟 LGB RMSE: {rmse:.4f}")

    # 特徴量重要度（Fold 0）
    if fold == 0:
        imp = lgb_model.feature_importances_
        n_snv = snv_tr.shape[1]
        n_d1 = d1_tr.shape[1]
        n_pca = 10

        cat = {}
        pos = 0
        cat['SNV'] = np.sum(imp[pos:pos+n_snv]); pos += n_snv
        cat['d1'] = np.sum(imp[pos:pos+n_d1]); pos += n_d1
        cat['PCA'] = np.sum(imp[pos:pos+n_pca]); pos += n_pca
        cat['KNN_mean'] = imp[pos]; pos += 1
        cat['ratio'] = imp[pos]; pos += 1
        cat['std'] = imp[pos]; pos += 1
        if PATTERN_A_ADD_D2 or PATTERN_C_BOTH:
            n_d2 = d2_tr.shape[1]
            cat['d2'] = np.sum(imp[pos:pos+n_d2]); pos += n_d2
        if PATTERN_B_KNN_DIST or PATTERN_C_BOTH:
            cat['KNN_dist'] = imp[pos]; pos += 1

        total = sum(cat.values())
        print(f"\n  📊 特徴量重要度:")
        for name, val in sorted(cat.items(), key=lambda x: -x[1]):
            pct = val / total * 100
            bar = '█' * int(pct)
            print(f"     {name:15s}: {val:6.0f} ({pct:5.1f}%) {bar}")


# ============================================================
# 3. 全体評価
# ============================================================
print(f"\n{'='*60}")
print("📊 全体評価")
print(f"{'='*60}")

y_true_real = np.expm1(y_train_log)
oof_rmse = np.sqrt(mean_squared_error(y_true_real, oof_lgb))

print(f"  🌟 LGB OOF RMSE: {oof_rmse:.4f}")
print(f"  📊 Fold平均 RMSE: {np.mean(fold_rmses):.4f} ± {np.std(fold_rmses):.4f}")

# 樹種別
print(f"\n  --- 樹種別残差 ---")
print(f"  {'樹種':12s} {'n':>4s} {'RMSE':>7s} {'bias':>7s}")
for sp in sorted(train['樹種'].unique()):
    mask = train['樹種'] == sp
    y_s = train.loc[mask, '含水率'].values
    p_s = oof_lgb[mask.values]
    rmse_s = np.sqrt(np.mean((y_s - p_s)**2))
    bias_s = np.mean(y_s - p_s)
    print(f"  {sp:12s} {len(y_s):4d} {rmse_s:7.2f} {bias_s:+7.2f}")


# ============================================================
# 4. 提出ファイル
# ============================================================
final_blend = np.clip(final_lgb, 0, None)

submit[1] = final_blend
out = f'submission_{pattern_name.replace(" ", "_")}.csv'
submit.to_csv(out, index=False, header=False)

print(f"\n✅ 提出ファイル: {out}")
print(f"📈 min={final_blend.min():.1f}%, median={np.median(final_blend):.1f}%, "
      f"max={final_blend.max():.1f}%")

print(f"\n📌 スコア比較:")
print(f"   元Blend:          LB = 12.647")
print(f"   LGB単独(元特徴量): LB = 12.615 ← 現BEST")
print(f"   LGB+加重KNN:      LB = 12.940 ← 悪化")
print(f"   今回({pattern_name}): LB = ???")

🧪 テスト: d2追加のみ

───────────────────────────────────────────────────────
📁 Fold 1/5  (train:940, valid:270)
   検証樹種: ['ウエンジ', 'トチ']

  📐 LGB入力次元: 4678
     ベース(元コード): 3123
     + d2:            1555
  🌟 LGB RMSE: 9.6703

  📊 特徴量重要度:
     d2             :   5112 ( 57.0%) █████████████████████████████████████████████████████████
     d1             :   2188 ( 24.4%) ████████████████████████
     SNV            :   1127 ( 12.6%) ████████████
     KNN_mean       :    411 (  4.6%) ████
     PCA            :    108 (  1.2%) █
     ratio          :     17 (  0.2%) 
     std            :      1 (  0.0%) 

───────────────────────────────────────────────────────
📁 Fold 2/5  (train:981, valid:229)
   検証樹種: ['チェリー', 'ヒノキ']
  🌟 LGB RMSE: 20.0253

───────────────────────────────────────────────────────
📁 Fold 3/5  (train:1009, valid:201)
   検証樹種: ['ウォールナット', 'クリ']
  🌟 LGB RMSE: 18.1867

───────────────────────────────────────────────────────
📁 Fold 4/5  (train:959, valid:251)
   検証樹種: ['ナラ', 'ベイマツ', '